# Incident Response Toolkit: walkthrough

This notebook walks through a flash flood scenario in Austin, TX, using the toolkit end to end.

* **Sections 1 and 2** are pure Python and were run before publishing, so their output is real.
* **Sections 3 to 6** need ArcGIS Pro 3.2 or later. Open this notebook in Pro, point the paths at your own geodatabase, and run them.

All incident data is synthetic. See `data/README.md` for public datasets to use with the ArcPy sections.

## 1. Validate the incident report before mapping anything

A dispatcher exported 30 incident rows. Five have problems that would silently put points in the wrong place or double count them.

In [ ]:
import sys
sys.path.insert(0, "../src")

from irtoolkit.incidents import load_incidents

result = load_incidents("../data/sample_incidents.csv")
print(result.summary())
print()
for line_no, reason in result.rejected:
    print(f"line {line_no:>3}: {reason}")

25 incidents loaded, 5 rows rejected

line  27: duplicate incident_id INC-0005
line  28: coordinates out of valid range (latitude and longitude look swapped)
line  29: coordinates are not numeric
line  30: outside response area
line  31: missing incident_id


Every rejected row has a specific reason, including the swapped latitude and longitude, one of the most common field data errors.

In [ ]:
from collections import Counter

for category, n in Counter(i.category for i in result.incidents).most_common():
    print(f"{category:<20} {n}")

ROAD CLOSURE         8
FLOODING             6
DOWNED POWER LINE    6
WATER RESCUE         5


## 2. Encode a data standard as rules

The shelter inventory standard lives in one place and is reused by the toolbox, notebooks, and scheduled jobs. Here it runs against one messy record.

In [ ]:
from datetime import date
from irtoolkit.qaqc_rules import evaluate
from irtoolkit.shelter_standard import SHELTER_RULES

record = {
    "OBJECTID": 7, "NAME": " Eastside Rec Center ", "STATUS": "full",
    "SHELTER_TYPE": "pet friendly", "CAPACITY": 200, "OCCUPANCY": 80,
    "LAST_INSPECTED": date(2010, 3, 1),
}
fixed, issues = evaluate(record, SHELTER_RULES)
for i in issues:
    fix = f"  -> fixed to '{i.fixed_to}'" if i.fixed_to is not None else ""
    print(f"[{i.rule}] {i.message}{fix}")

[name spacing] leading or trailing spaces  -> fixed to 'Eastside Rec Center'
[status domain] 'full' should be 'Full'  -> fixed to 'Full'
[type domain] 'pet friendly' should be 'Pet Friendly'  -> fixed to 'Pet Friendly'
[inspection date] 2010-03-01 is outside 2015-01-01 to 2026-11-24
[full status check] status is Full but occupancy is under 90 percent of capacity


Formatting problems are fixed automatically. Judgment calls, like a shelter marked Full at 40 percent occupancy, are flagged for a person to review.

## 3. Profile and stage data with `arcpy.da.Describe` (ArcGIS Pro)

In [ ]:
from irtoolkit.gis.describe import clone_empty, dataset_profile

GDB = r"C:\GIS\FloodResponse\FloodResponse.gdb"   # change to your geodatabase
print(dataset_profile(f"{GDB}\\AddressPoints"))

# Empty copy with the same schema and coordinate system, for incoming field edits
clone_empty(f"{GDB}\\AddressPoints", out_name="AddressPoints_staging")

## 4. Standardize partner data with list functions and loops (ArcGIS Pro)

Project everything a partner sent into Texas Centric Albers Equal Area (WKID 3083), skipping layers that are already correct.

In [ ]:
from irtoolkit.gis.batch import batch_project, inventory

PARTNER = r"C:\GIS\FloodResponse\PartnerDelivery.gdb"
inventory(PARTNER)
report = batch_project(PARTNER, GDB, target_sr=3083)
print(report.summary())

## 5. Map the impact area and build the notification list (ArcGIS Pro)

Geometry objects do the work in memory: incident points become a multipoint, then a buffered convex hull. Cursors copy every address inside it to a notification layer.

In [ ]:
import arcpy
from irtoolkit.gis.impact import affected_features, impact_area

area = impact_area(result.incidents, buffer_distance=500, work_wkid=3083)  # meters
arcpy.management.CopyFeatures(area, f"{GDB}\\ImpactArea")
affected_features(area, f"{GDB}\\AddressPoints", f"{GDB}\\NotifyAddresses",
                  keep_fields=["FULL_ADDRESS", "ZIP"])

## 6. Run QA/QC on a live layer (ArcGIS Pro)

Read mode first. Apply fixes inside an edit session only after the report has been reviewed.

In [ ]:
from irtoolkit.gis.qaqc import run_qaqc

issues = run_qaqc(f"{GDB}\\Shelters", SHELTER_RULES, "shelter_issues.csv", apply_fixes=False)
print(len(issues), "issues")